In [ ]:
# ---------------- Imports ----------------
import json
import random
from collections import defaultdict
from pathlib import Path
import os
import re

import pandas as pd
import yaml
import csv



In [ ]:
# ---------------- Args ----------------
input_files = [
    "20260115T095923-combined-claims-full.shard0",
    "20260115T100139-combined-claims-full.shard1",
    "20260115T095935-combined-claims-full.shard2",
    
]

index_name = "combined-claims-full-index"

SHARD_SIZE = 100_000
TRAIN_PCT = 0.8
DEV_PCT = 0.1

RANDOM_SEED = 42


In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
DATA_FOLDER = os.path.join(PROJ_STORE, "data")


INDEX_PATH = os.path.join(DATA_FOLDER, "index", f"{index_name}.json")

INPUT_DIR = os.path.join(DATA_FOLDER, "augmented-raw")
#input_path = os.path.join(INPUT_DIR, f"{input_file}.jsonl")
INPUT_FILES = [
    Path(INPUT_DIR) / f"{name}.jsonl"
    for name in input_files
]
missing = [p for p in INPUT_FILES if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing input files:\n" +
        "\n".join(str(p) for p in missing)
    )


# OUTPUT
OUTPUT_DIR = os.path.join(DATA_FOLDER, "augmented-processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)
first_name = input_files[0]
base_name = re.sub(r"\.shard\d+$", "", first_name)
output_path = os.path.join(OUTPUT_DIR, f"{base_name}")

STATS_OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "dataset-stats", f"{base_name}")
os.makedirs(STATS_OUTPUT_DIR, exist_ok=True)



ALLOWED_FIELDS = [
    "claim_id",
    #"claim_text",
    "restated_claim",
    "framing_type",
    "true_label",
]

with open(INDEX_PATH, "r") as f:
    WIKI_INDEX = json.load(f)
        

In [ ]:

def write_verification_stats_csv(input_files, output_path):

    stats = defaultdict(lambda: {
        "true": 0,
        "false": 0,
        "missing": 0,
        "total": 0,
    })

    total = {
        "true": 0,
        "false": 0,
        "missing": 0,
        "total": 0,
    }

    for path in input_files:
        print(f"Scanning {path}")

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                row = json.loads(line)

                ft = row.get("framing_type", "MISSING")
                vp = row.get("verification_passed")

                # Normalize framing_type
                if ft is None:
                    ft = "MISSING"
                else:
                    ft = str(ft).strip()

                # Count
                if vp is True:
                    stats[ft]["true"] += 1
                    total["true"] += 1

                elif vp is False:
                    stats[ft]["false"] += 1
                    total["false"] += 1

                else:
                    stats[ft]["missing"] += 1
                    total["missing"] += 1

                stats[ft]["total"] += 1
                total["total"] += 1

    csv_path = os.path.join(output_path, f"{base_name}-verification-stats.csv")

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Header
        writer.writerow([
            "framing_type",
            "verification_true",
            "verification_false",
            "verification_missing",
            "total"
        ])

        # Rows per framing type
        for ft in sorted(stats):
            row = stats[ft]
            writer.writerow([
                ft,
                row["true"],
                row["false"],
                row["missing"],
                row["total"],
            ])

        # TOTAL row
        writer.writerow([
            "TOTAL",
            total["true"],
            total["false"],
            total["missing"],
            total["total"],
        ])

    print(f"Wrote verification stats to: {csv_path}")


# Create verification stats CSV (before filtering)
write_verification_stats_csv(INPUT_FILES, STATS_OUTPUT_DIR)


In [ ]:
# ---------------- Functions ----------------

def load_jsonl_files(paths):
    rows = []

    for path in paths:
        print(f"Loading {path}")
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))

    print(f"Loaded {len(rows)} total rows")
    return rows

def base_claim_id(claim_id: str) -> str:
    return claim_id.split(":", 1)[0]

def keep_row(row):
    # Always keep originals
    if row.get("framing_type") == "original":
        return True

    # Keep only verified restatements
    return row.get("verification_passed") is True


def write_grouped_shards(claims, output_dir, split_name, shard_size):
    os.makedirs(output_dir, exist_ok=True)

    shard = []
    shard_rows = 0
    shard_id = 0

    for variants in claims.values():

        if shard_rows + len(variants) > shard_size and shard:
            out_path = os.path.join(
                output_dir,
                f"{split_name}-{shard_id:04d}.jsonl"
            )

            with open(out_path, "w", encoding="utf-8") as f:
                for r in shard:
                    f.write(json.dumps(r) + "\n")

            shard_id += 1
            shard = []
            shard_rows = 0

        shard.extend(variants)
        shard_rows += len(variants)

    if shard:
        out_path = os.path.join(
            output_dir,
            f"{split_name}-{shard_id:04d}.jsonl"
        )

        with open(out_path, "w", encoding="utf-8") as f:
            for r in shard:
                f.write(json.dumps(r) + "\n")


    
def normalize_and_strip_row(row):
    out = {}

    # Normalize fields
    for k in ALLOWED_FIELDS:
        if k == "restated_claim":
            if row.get("framing_type") == "original":
                out["restated_claim"] = row["claim_text"]
            else:
                out["restated_claim"] = row.get("restated_claim")
        elif k in row:
            out[k] = row[k]

    # Add messages field
    out["messages"] = [
        {"role": "user", "content": out["restated_claim"]},
        {"role": "assistant", "content": out["true_label"]},
    ]

    return out



def build_source_groups(claims, wiki_index):


    page_to_claims = defaultdict(set)

    # Build reverse index: page -> claims
    for cid in claims.keys():

        full_id = cid

        pages = wiki_index.get(full_id, [])

        for p in pages:
            page_to_claims[p].add(cid)

    visited = set()
    groups = []

    # BFS over connected components
    for cid in claims.keys():

        if cid in visited:
            continue

        stack = [cid]
        group = set()

        while stack:
            cur = stack.pop()

            if cur in visited:
                continue

            visited.add(cur)
            group.add(cur)

            pages = wiki_index.get(cur, [])

            for p in pages:
                for other in page_to_claims[p]:
                    if other not in visited:
                        stack.append(other)

        groups.append(group)

    return groups



In [ ]:
# ---------------- Workspace ----------------

def split_claims_keep_variants(input_files, output_dir, seed):
    
    rows = load_jsonl_files(input_files)

    # 1) Filter failed verification
    rows = [r for r in rows if keep_row(r)]
    
    # 1.5) Normalize + strip fields
    rows = [normalize_and_strip_row(r) for r in rows]

    # 2) Group by base claim
    claims = defaultdict(list)
    for r in rows:
        claims[base_claim_id(r["claim_id"])].append(r)

    claim_ids = list(claims.keys())

    # 3) Build source-aware units
    groups = build_source_groups(claims, WIKI_INDEX)
    
    
    sizes = [
    sum(len(claims[cid]) for cid in g)
    for g in groups
]

    sizes.sort(reverse=True)

    print("Number of groups:", len(groups))
    print("Largest groups:", sizes[:10])
    print("Total rows:", sum(sizes))
        
    
            
    # Compute group sizes
    group_sizes = [(g, sum(len(claims[c]) for c in g)) for g in groups]
    group_sizes.sort(key=lambda x: x[1])  # smallest-first

    total_rows = sum(size for _, size in group_sizes)
    dev_target = int(total_rows * DEV_PCT)
    test_target = int(total_rows * (1 - TRAIN_PCT - DEV_PCT))

    train_groups = []
    dev_groups = []
    test_groups = []

    dev_rows = 0
    test_rows = 0

    # biggest = max(group_sizes, key=lambda x: x[1])
    # train_groups.append(biggest[0])
    # group_sizes.remove(biggest)

    for g, size in group_sizes:
        if dev_rows + size <= dev_target:
            dev_groups.append(g)
            dev_rows += size
        elif test_rows + size <= test_target:
            test_groups.append(g)
            test_rows += size
        else:
            train_groups.append(g)

    if dev_rows < dev_target:
        # move the smallest remaining train group into dev
        if train_groups:
            g = train_groups.pop(0)  # NOTE: only works if train_groups is still smallest-first ordered
            s = sum(len(claims[c]) for c in g)
            dev_groups.append(g)
            dev_rows += s

    if test_rows < test_target:
        if train_groups:
            g = train_groups.pop(0)
            s = sum(len(claims[c]) for c in g)
            test_groups.append(g)
            test_rows += s

    # Print achieved sizes
    def count_groups(gs):
        return sum(sum(len(claims[c]) for c in g) for g in gs)

    train_rows = count_groups(train_groups)
    dev_rows = count_groups(dev_groups)
    test_rows = count_groups(test_groups)
    final_total = train_rows + dev_rows + test_rows

    print("\nFinal split rows:")
    print("Train:", train_rows, f"({train_rows/final_total:.2%})")
    print("Dev:  ", dev_rows,   f"({dev_rows/final_total:.2%})")
    print("Test: ", test_rows,  f"({test_rows/final_total:.2%})")
    
    

    
    
    


    splits = {
        "train": {},
        "dev": {},
        "test": {},
    }


    def assign(groups, split_name):
        for g in groups:
            for cid in g:
                splits[split_name][cid] = claims[cid]


    assign(train_groups, "train")
    assign(dev_groups, "dev")
    assign(test_groups, "test")



    for split, split_claims in splits.items():

        split_dir = os.path.join(output_dir, split)

        write_grouped_shards(
            claims=split_claims,
            output_dir=split_dir,
            split_name=split,
            shard_size=SHARD_SIZE,
        )





In [ ]:
# -------------------------
# Call
# -------------------------

if __name__ == "__main__":
    split_claims_keep_variants(
        input_files=INPUT_FILES,
        output_dir=output_path,
        seed=RANDOM_SEED,
    )

